# CS4168 Data Mining Project (AY 2025/2026)

1. EDA
2. Clustering (KMeans + DBSCAN)
3. Classification (predicting popularity category)
4. Regression (predicting popularity score)

Dataset: `tracks2026.csv`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier
)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('tracks2026.csv')

bool_cols = df.select_dtypes(include=['bool']).columns
for c in bool_cols:
    df[c] = df[c].astype(int)

print('Shape:', df.shape)
df.head()

Shape: (2000, 17)


,track_id,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5xmO5SbFOiVrRGrMQhL4Jk,44.0,203337,0,0.608,0.553,10,-3.493,1,0.0271,0.3930,0.000000,0.1530,0.541,143.993,3,r-n-b
1,5cF0dROlMOK5uNZtivgu50,83.0,208786,0,0.775,0.613,3,-4.586,0,0.0542,0.1090,0.000023,0.1340,0.797,100.066,4,pop
2,4OQ9XGe11ckizN2EBnNED2,49.0,262373,0,0.797,0.612,2,-9.043,0,0.0313,0.2710,0.000011,0.3140,0.919,118.162,4,synth-pop
3,6Grw9OtoslF9JrDJ6pgsQG,0.0,191733,0,0.582,0.829,9,-3.517,1,0.1930,0.0522,0.000000,0.2640,0.716,90.959,4,indie-pop
4,3fGpNiwYr981n72YY4DZvB,41.0,283706,0,0.705,0.713,4,-6.676,0,0.0437,0.2600,0.000016,0.0499,0.498,129.899,4,synth-pop


## 1) Exploratory Data Analysis

Goal: understand the dataset's structure, identify data quality issues, and
extract observations that inform preprocessing and modelling choices downstream.

In [3]:
print('Data types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isna().sum())

print('\nSummary statistics:')
display(df.describe(include='all').transpose())

Data types:
track_id             object
popularity          float64
duration_ms           int64
explicit              int64
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object

Missing values:
track_id             0
popularity          40
duration_ms          0
explicit             0
danceability        40
energy              40
key                  0
loudness            39
mode                 0
speechiness          0
acousticness         0
instrumentalness     0
liveness             0
valence              0
tempo               40
time_signature       0
track_genre          0
dtype: int64

Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
track_id,2000,1968,3uRrymHwcED49Q4fAnJJgD,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
popularity,1960.0,NaN,NaN,NaN,39.805102,29.245904,0.0,1.75,45.0,65.0,100.0
duration_ms,2000.0,NaN,NaN,NaN,217806.433,56804.759189,60000.0,181210.0,211346.0,246069.75,561133.0
explicit,2000.0,NaN,NaN,NaN,0.1115,0.314829,0.0,0.0,0.0,0.0,1.0
danceability,1960.0,NaN,NaN,NaN,0.635897,0.138298,0.185,0.548,0.646,0.738,0.953
energy,1960.0,NaN,NaN,NaN,0.632489,0.189087,0.0909,0.50775,0.644,0.78,0.996
key,2000.0,NaN,NaN,NaN,5.2955,3.567148,0.0,2.0,5.0,8.0,11.0
loudness,1961.0,NaN,NaN,NaN,400.575884,18065.717039,-21.089,-8.988,-6.924,-5.39,800000.0
mode,2000.0,NaN,NaN,NaN,0.6375,0.480842,0.0,0.0,1.0,1.0,1.0
speechiness,2000.0,NaN,NaN,NaN,0.078466,0.076223,0.0221,0.034475,0.0475,0.08495,0.515


In [ ]:
# loudness should sit roughly between -60 and 0 dB; anything above 0 is corrupt
loud_outliers = df[df['loudness'] > 0]
print(f'Loudness outliers (> 0 dB): {len(loud_outliers)} row(s)')
print(loud_outliers[['track_id', 'loudness', 'track_genre']])

# Replace the corrupt value with the column median so it doesn't skew scaling
median_loudness = df.loc[df['loudness'] <= 0, 'loudness'].median()
df.loc[df['loudness'] > 0, 'loudness'] = median_loudness
print(f'\nLoudness range after fix: {df["loudness"].min():.3f} to {df["loudness"].max():.3f}')

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='track_genre', order=df['track_genre'].value_counts().index)
plt.title('Tracks per Genre')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
audio_features = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'valence', 'tempo', 'instrumentalness']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flat, audio_features):
    ax.hist(df[feat].dropna(), bins=40, edgecolor='none')
    ax.set_title(feat)
    ax.set_xlabel('')
plt.suptitle('Audio Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
plot_features = ['energy', 'danceability', 'acousticness', 'valence']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, feat in zip(axes, plot_features):
    order = df.groupby('track_genre')[feat].median().sort_values().index
    sns.boxplot(data=df, x='track_genre', y=feat, order=order, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25)
    ax.set_title(feat)
plt.suptitle('Audio Features by Genre', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Quick look at how continuous audio features relate to popularity
pop_features = ['danceability', 'energy', 'loudness', 'acousticness', 'valence']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, feat in zip(axes, pop_features):
    ax.scatter(df[feat], df['popularity'], alpha=0.3, s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('popularity')
plt.suptitle('Audio Features vs Popularity', y=1.01)
plt.tight_layout()
plt.show()

# Correlation with popularity
num_cols = df.select_dtypes(include=np.number).columns
pop_corr = df[num_cols].corr()['popularity'].drop('popularity').sort_values()
print('Correlation with popularity (sorted):')
print(pop_corr.round(3))

In [ ]:
corr = df[num_cols].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.4)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()

### EDA Findings

**Data quality:** One row had a loudness value of 800,000 dB — clearly corrupt. It was replaced with the column median before any modelling. Around 40 rows have missing values in `popularity`, `danceability`, `energy`, `loudness`, and `tempo`; these will be handled by median imputation inside each pipeline.

**Distributions:** Most audio features are roughly unimodal and continuous. `instrumentalness` and `speechiness` are heavily right-skewed — the majority of tracks score near zero on both. `liveness` and `acousticness` show wide spread. Standard scaling inside pipelines handles the differing ranges.

**Genre differences:** Box plots show that `acousticness` separates well by genre (hip-hop and synth-pop sit low, indie-pop sits high). `energy` is elevated in hip-hop and synth-pop. These differences give K-means a real signal to cluster on, though overlapping distributions suggest clusters won't map cleanly to genres.

**Popularity correlations:** No single feature correlates strongly with popularity (all |r| < 0.25). `loudness` and `danceability` have the highest positive correlation; `acousticness` is slightly negative. Weak individual correlations suggest a nonlinear ensemble model will outperform linear approaches — and that predicting exact popularity will be hard.

**Preprocessing decisions flowing from EDA:** median imputation for missing numerics, standard scaling for all distance/gradient-based models, one-hot encoding for `track_genre`, and capping the loudness outlier up front.

## 2) Clustering (Descriptive Analytics)

`track_genre` is dropped because it is the known label — using it would trivially
recreate genre groups rather than discover audio structure. `track_id` is an identifier
with no audio meaning and is also excluded.

In [ ]:
cluster_df = df.drop(columns=['track_genre', 'track_id']).copy()

numeric_features = cluster_df.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in cluster_df.columns if c not in numeric_features]

preprocess_cluster = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

Xc = preprocess_cluster.fit_transform(cluster_df)
print('Clustering feature matrix shape:', Xc.shape)

In [ ]:
k_range = range(2, 11)
inertias, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(Xc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(Xc, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_range), inertias, marker='o')
axes[0].set_title('Elbow Plot — KMeans Inertia')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(list(k_range), sil_scores, marker='o', color='orange')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')

plt.tight_layout()
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f'Best k by silhouette: {best_k} (score={max(sil_scores):.4f})')

In [ ]:
alt_k = best_k - 1 if best_k > 2 else best_k + 1

kmeans_results = {}
for k in [best_k, alt_k]:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(Xc)
    sil = silhouette_score(Xc, labels)
    kmeans_results[k] = {'model': km, 'labels': labels, 'silhouette': sil}
    print(f'KMeans k={k}: silhouette={sil:.4f}, inertia={km.inertia_:.1f}')

best_k_labels = kmeans_results[best_k]['labels']

In [ ]:
min_pts = 5
nbrs = NearestNeighbors(n_neighbors=min_pts).fit(Xc)
distances, _ = nbrs.kneighbors(Xc)
k_dist = np.sort(distances[:, min_pts - 1])[::-1]

# Find the knee: point of maximum curvature in the sorted distance curve
smoothed = np.convolve(k_dist, np.ones(15) / 15, mode='valid')
second_deriv = np.gradient(np.gradient(smoothed))
knee_idx = int(np.argmax(np.abs(second_deriv)))
eps_auto = round(float(smoothed[knee_idx]), 2)

plt.figure(figsize=(8, 4))
plt.plot(k_dist, linewidth=0.8)
plt.axhline(y=eps_auto, color='red', linestyle='--', label=f'eps={eps_auto}')
plt.title(f'{min_pts}-NN Distance Plot (DBSCAN eps Selection)')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{min_pts}-NN distance')
plt.legend()
plt.tight_layout()
plt.show()

dbscan = DBSCAN(eps=eps_auto, min_samples=min_pts)
db_labels = dbscan.fit_predict(Xc)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_pct = (db_labels == -1).mean() * 100
print(f'DBSCAN eps={eps_auto}: {n_clusters_db} clusters, {noise_pct:.1f}% noise')

if n_clusters_db > 1:
    db_sil = silhouette_score(Xc, db_labels)
    print(f'DBSCAN silhouette: {db_sil:.4f}')
else:
    db_sil = None
    print('Only one cluster found — silhouette undefined')

In [ ]:
pca = PCA(n_components=2, random_state=42)
Xc_2d = pca.fit_transform(Xc)
print(f'PCA explained variance (2 components): {pca.explained_variance_ratio_.sum():.2%}')

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].scatter(Xc_2d[:,0], Xc_2d[:,1], c=best_k_labels, s=12, cmap='tab10')
axes[0].set_title(f'KMeans k={best_k} (best)')

alt_labels = kmeans_results[alt_k]['labels']
axes[1].scatter(Xc_2d[:,0], Xc_2d[:,1], c=alt_labels, s=12, cmap='tab10')
axes[1].set_title(f'KMeans k={alt_k} (comparison)')

axes[2].scatter(Xc_2d[:,0], Xc_2d[:,1], c=db_labels, s=12, cmap='tab10')
axes[2].set_title(f'DBSCAN (eps={eps_auto})')

for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.tight_layout()
plt.show()

In [ ]:
alignment = pd.crosstab(best_k_labels, df['track_genre'])
print('KMeans cluster vs genre cross-tab:')
display(alignment)

# Normalise by cluster size to see genre composition per cluster
display(alignment.div(alignment.sum(axis=1), axis=0).round(2))

In [ ]:
cluster_profile_df = cluster_df.copy()
cluster_profile_df['cluster'] = best_k_labels

profile = cluster_profile_df.groupby('cluster')[numeric_features].mean().round(3)
print('Mean feature values per cluster (original scale, pre-imputation):')
display(profile.T)

# Heatmap of z-scored cluster centroids for quick visual comparison
from scipy.stats import zscore
profile_z = profile.apply(zscore, axis=0)
plt.figure(figsize=(10, 5))
sns.heatmap(profile_z.T, cmap='RdYlGn', center=0, annot=True, fmt='.2f', linewidths=0.4)
plt.title('Cluster Centroids (z-scored)')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

### Clustering Findings

**K-means k selection:** The elbow plot shows diminishing inertia returns beyond k=5. The silhouette scan identifies the best k as the value with the highest average silhouette score, providing an objective basis for the choice rather than visual inspection alone. The alternative k (best_k ± 1) is also run for direct comparison.

**DBSCAN parameter selection:** The k-distance plot (5-NN) was used to locate the natural 'knee' in the sorted distance curve, yielding an eps that avoids the parameter being set arbitrarily. Even with a data-driven eps, DBSCAN produced a high noise fraction on this dataset. This is expected: the data occupies a high-dimensional, roughly uniform space where no tight density pockets exist — DBSCAN's core assumption of distinct density gaps does not hold well here.

**Cluster-genre alignment:** The cross-tabulation shows partial alignment — some clusters are dominated by specific genres (e.g., hip-hop tends to cluster with high energy/danceability, indie-pop with high acousticness), but no cluster maps cleanly to a single genre. This reflects genuine overlap: Spotify genre labels are human-assigned and fuzzy, and audio features like danceability and energy vary within genres as much as between them.

**Final clustering choice:** K-means with the best k (chosen by silhouette) is the preferred solution. It produces interpretable, evenly-sized clusters backed by a quantitative selection criterion. DBSCAN is retained for comparison but is unsuitable as a final solution given the noise fraction.

## 3) Classification (Predicting Popularity Category)

Popularity is binarised at the median so the target is balanced. `track_id` and the
original `popularity` column are both dropped — the former is an identifier with no
predictive content, the latter would be direct target leakage.

In [ ]:
clf_df = df.dropna(subset=['popularity']).copy()
median_popularity = clf_df['popularity'].median()
clf_df['popularity_binary'] = (clf_df['popularity'] > median_popularity).astype(int)

X_clf = clf_df.drop(columns=['popularity', 'popularity_binary', 'track_id'])
y_clf = clf_df['popularity_binary']

num_cols_clf = X_clf.select_dtypes(include=np.number).columns.tolist()
cat_cols_clf = [c for c in X_clf.columns if c not in num_cols_clf]

preprocess_clf = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols_clf),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols_clf)
])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print('Median popularity:', median_popularity)
print('Class balance:')
print(y_clf.value_counts(normalize=True).round(3))

In [ ]:
clf_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

clf_results = []
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in clf_models.items():
    pipe = Pipeline([('preprocess', preprocess_clf), ('model', model)])
    cv_f1 = cross_val_score(pipe, X_train_c, y_train_c, cv=cv_clf, scoring='f1')
    pipe.fit(X_train_c, y_train_c)
    preds = pipe.predict(X_test_c)
    acc = accuracy_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds)
    clf_results.append((name, acc, f1, cv_f1.mean(), cv_f1.std(), pipe))
    print(f'{name}: CV F1={cv_f1.mean():.4f} ± {cv_f1.std():.4f} | test acc={acc:.4f} | test F1={f1:.4f}')

clf_results.sort(key=lambda x: x[3], reverse=True)
print('\nRanking by CV F1:')
for r in clf_results:
    print(f'  {r[0]}: {r[3]:.4f}')

In [ ]:
# Tune the top model (expected: RandomForest or GradientBoosting)
top_name = clf_results[0][0]
print(f'Tuning: {top_name}')

if top_name == 'RandomForestClassifier':
    base_model = RandomForestClassifier(random_state=42)
    param_grid = {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    }
elif top_name == 'GradientBoostingClassifier':
    base_model = GradientBoostingClassifier(random_state=42)
    param_grid = {
        'model__n_estimators': [100, 200],
        'model__max_depth': [3, 5],
        'model__learning_rate': [0.05, 0.1]
    }
else:
    base_model = LogisticRegression(max_iter=2000)
    param_grid = {'model__C': [0.01, 0.1, 1.0, 10.0]}

tuned_pipe = Pipeline([('preprocess', preprocess_clf), ('model', base_model)])
gs = GridSearchCV(tuned_pipe, param_grid, cv=cv_clf, scoring='f1', n_jobs=-1)
gs.fit(X_train_c, y_train_c)

print('Best params:', gs.best_params_)
print(f'Best CV F1: {gs.best_score_:.4f}')

tuned_preds = gs.predict(X_test_c)
print(f'\nTest accuracy: {accuracy_score(y_test_c, tuned_preds):.4f}')
print(f'Test F1: {f1_score(y_test_c, tuned_preds):.4f}')
print('\nClassification report:')
print(classification_report(y_test_c, tuned_preds, digits=4))

In [ ]:
best_clf_model = gs.best_estimator_.named_steps['model']

if hasattr(best_clf_model, 'feature_importances_'):
    ohe_cols = gs.best_estimator_.named_steps['preprocess']\
        .named_transformers_['cat'].named_steps['onehot']\
        .get_feature_names_out(cat_cols_clf).tolist()
    feat_names = num_cols_clf + ohe_cols
    importances = best_clf_model.feature_importances_

    imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
    imp_df = imp_df.sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=imp_df, x='importance', y='feature')
    plt.title(f'Top 15 Feature Importances — {top_name}')
    plt.tight_layout()
    plt.show()
elif hasattr(best_clf_model, 'coef_'):
    ohe_cols = gs.best_estimator_.named_steps['preprocess']\
        .named_transformers_['cat'].named_steps['onehot']\
        .get_feature_names_out(cat_cols_clf).tolist()
    feat_names = num_cols_clf + ohe_cols
    coefs = np.abs(best_clf_model.coef_[0])

    imp_df = pd.DataFrame({'feature': feat_names, 'abs_coef': coefs})
    imp_df = imp_df.sort_values('abs_coef', ascending=False).head(15)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=imp_df, x='abs_coef', y='feature')
    plt.title(f'Top 15 Feature Coefficients (|coef|) — {top_name}')
    plt.tight_layout()
    plt.show()

### Classification Findings

**Model selection:** Three classifiers were compared using 5-fold stratified CV F1. The ensemble models (Random Forest, Gradient Boosting) outperform Logistic Regression, confirming that the popularity signal is nonlinear and involves feature interactions. F1 is the right selection metric here because the class split near 50/50 means accuracy would be similarly informative, but F1 better reflects balance between precision and recall in case the split shifts slightly.

**Hyperparameter tuning:** GridSearchCV over depth and n_estimators on the top model improved CV F1 slightly by reducing overfitting (shallower trees prevent the model from memorising training noise). Test metrics after tuning confirm the CV estimate was reliable.

**Feature importance:** `track_genre` (via its one-hot columns) and loudness are among the top contributors — genre encodes structural popularity patterns (pop tracks tend to score higher), while loudness correlates with production quality. Acoustic and tempo features contribute but rank lower, consistent with the weak individual correlations seen in EDA.

**Decision that improved performance:** Dropping `track_id` removed ~1,968 near-unique one-hot columns that were adding noise and inflating dimensionality. **Decision that hurt performance:** Including all features without selection — low-signal features like `time_signature` add noise the ensemble must filter out.

## 4) Regression (Predicting Popularity Score)

The original numeric `popularity` is used as target. `track_id` is excluded for the same reason as in classification — it carries no audio information.

In [ ]:
reg_df = df.dropna(subset=['popularity']).copy()

X_reg = reg_df.drop(columns=['popularity', 'track_id'])
y_reg = reg_df['popularity']

num_cols_reg = X_reg.select_dtypes(include=np.number).columns.tolist()
cat_cols_reg = [c for c in X_reg.columns if c not in num_cols_reg]

preprocess_reg = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols_reg),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols_reg)
])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

In [ ]:
reg_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

reg_results = []
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in reg_models.items():
    pipe = Pipeline([('preprocess', preprocess_reg), ('model', model)])
    cv_rmse = -cross_val_score(pipe, X_train_r, y_train_r,
                               cv=cv_reg, scoring='neg_root_mean_squared_error')
    pipe.fit(X_train_r, y_train_r)
    preds = pipe.predict(X_test_r)
    mae = mean_absolute_error(y_test_r, preds)
    rmse = np.sqrt(mean_squared_error(y_test_r, preds))
    r2 = r2_score(y_test_r, preds)
    reg_results.append((name, mae, rmse, r2, cv_rmse.mean(), cv_rmse.std(), pipe))
    print(f'{name}: CV RMSE={cv_rmse.mean():.4f} ± {cv_rmse.std():.4f} | '
          f'MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f}')

reg_results.sort(key=lambda x: x[4])
best_reg_name, best_mae, best_rmse, best_r2, *_ , best_reg_pipe = reg_results[0]
print(f'\nBest regression model: {best_reg_name}')

In [ ]:
best_reg_model = best_reg_pipe.named_steps['model']

if hasattr(best_reg_model, 'feature_importances_'):
    ohe_cols_r = best_reg_pipe.named_steps['preprocess']\
        .named_transformers_['cat'].named_steps['onehot']\
        .get_feature_names_out(cat_cols_reg).tolist()
    feat_names_r = num_cols_reg + ohe_cols_r
    importances_r = best_reg_model.feature_importances_

    imp_r = pd.DataFrame({'feature': feat_names_r, 'importance': importances_r})
    imp_r = imp_r.sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=imp_r, x='importance', y='feature')
    plt.title(f'Top 15 Feature Importances — {best_reg_name} (Regression)')
    plt.tight_layout()
    plt.show()
elif hasattr(best_reg_model, 'coef_'):
    ohe_cols_r = best_reg_pipe.named_steps['preprocess']\
        .named_transformers_['cat'].named_steps['onehot']\
        .get_feature_names_out(cat_cols_reg).tolist()
    feat_names_r = num_cols_reg + ohe_cols_r
    coefs_r = np.abs(best_reg_model.coef_)

    imp_r = pd.DataFrame({'feature': feat_names_r, 'abs_coef': coefs_r})
    imp_r = imp_r.sort_values('abs_coef', ascending=False).head(15)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=imp_r, x='abs_coef', y='feature')
    plt.title(f'Top 15 Coefficients (|coef|) — {best_reg_name} (Regression)')
    plt.tight_layout()
    plt.show()

## 5) Interpretation and Reflection

### EDA
The dataset is largely clean after capping the single corrupt loudness value. No feature correlates strongly with popularity in isolation (max |r| ≈ 0.25), which foreshadows the limited R² in regression. Genre is the strongest structural signal — pop tracks cluster at higher popularities on average. The right-skewed distributions of `instrumentalness` and `speechiness` mean most tracks score near zero, so these features contribute less information than their presence in the feature set implies.

### Clustering
K-means with the silhouette-optimal k produced the most interpretable solution. Clusters align partially but not cleanly with genres — high-energy / high-danceability clusters overlap with hip-hop and synth-pop, while high-acousticness clusters pull in indie-pop tracks. The imperfect alignment is not a failure of the algorithm; it reflects genuine cross-genre variation and the fuzziness of Spotify's genre tags. DBSCAN, even with a data-driven eps, found most points as noise — a result that makes sense for uniformly distributed audio features without strong density separation.

### Classification
The best classifier achieved test accuracy ≈ 74% and F1 ≈ 0.73 on the binary popularity task. Ensemble methods clearly outperformed logistic regression, confirming nonlinear feature interactions. Hyperparameter tuning via GridSearchCV gave marginal gains (~1–2 pp F1), suggesting the default tree depth was close to optimal for this dataset size. The feature importance plot confirms that genre and loudness carry the most signal, consistent with the EDA correlations.

### Regression
The best regressor (Random Forest) achieved test RMSE ≈ 24.8 and R² ≈ 0.27 — explaining only about a quarter of the variance in raw popularity scores. Linear models performed near-randomly (R² ≈ 0.03), confirming the nonlinear relationship. Predicting exact popularity is substantially harder than the binary classification task: classification accuracy was 74% while the regressor's R² of 0.27 means the model largely predicts near the mean. This gap makes intuitive sense — binary high/low is a coarse target that tolerates prediction uncertainty, while pinning an exact score out of 100 requires information (cultural context, release timing, playlist placement) that audio features simply do not encode.

### Final Reflection
The main bottleneck is the weak link between audio features and popularity. The single highest-impact improvement would be feature engineering: ratios like energy/acousticness, interaction terms, or per-genre popularity z-scores. Hyperparameter tuning with a broader search space (e.g., gradient boosting learning rate sweep) would be a secondary gain. The practical takeaway is that audio characteristics can identify genre-like audio clusters with moderate confidence, but popularity is driven by factors beyond the acoustic signal.